# 자식 청크로 찾고 부모 문서를 반환하기

이 노트북은 공개 PDF 6건과 Ch11의 30문항을 사용합니다. 첫 셀부터 순서대로 실행합니다.


## 입력 준비

검색 점수는 자식 청크에서 계산하고, 반환 단계에서 parent_id로 원문 전체를 찾습니다.


In [1]:
from pathlib import Path
import json
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "labs/part3/ch12-retrieval-tuning").exists())
data = json.loads((root / "labs/part3/ch12-retrieval-tuning/data/advanced-documents.json").read_text())
queries = json.loads((root / "labs/part3/ch12-retrieval-tuning/data/advanced-queries.json").read_text())
chunks, parents = data["chunks"], {item["doc_id"]: item for item in data["parents"]}
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=1)
matrix = vectorizer.fit_transform([item["text"] for item in chunks])
print(f"부모 문서 {len(parents)}건 · 자식 청크 {len(chunks)}개 · 평가 질문 {len(queries)}개")

부모 문서 6건 · 자식 청크 175개 · 평가 질문 30개


## 작은 성공 예제

한 질문에서 가장 가까운 자식 청크와 그 부모 문서를 나란히 봅니다.


In [2]:
question = next(item for item in queries if item["qid"] == "Q001")
assert question["split"] == "dev"
scores = cosine_similarity(vectorizer.transform([question["question"]]), matrix)[0]
best = int(np.argmax(scores))
child = chunks[best]
parent = parents[child["parent_id"]]
print("자식:", child["chunk_id"], child["text"][:180])
print("부모:", parent["doc_id"], parent["title"], "글자 수", len(parent["text"]))

자식: E01-P01-C01 미르시립도서관 이용 안내 미르시립도서관 2026년판 발행 2026. 3. 1.
부모: E01 미르시립도서관 이용 안내 글자 수 3020


## 직접 구현

여러 자식이 같은 부모를 가리킬 때 부모는 한 번만 반환합니다.


In [3]:
def parent_document_search(question, child_k=8, parent_k=3):
    scores = cosine_similarity(vectorizer.transform([question]), matrix)[0]
    result, seen = [], set()
    for idx in np.argsort(-scores)[:child_k]:
        parent_id = chunks[idx]["parent_id"]
        if parent_id not in seen:
            result.append(parents[parent_id])
            seen.add(parent_id)
        if len(result) == parent_k:
            break
    return result

for item in parent_document_search(question["question"]):
    print(item["doc_id"], item["title"], len(item["text"]))

E01 미르시립도서관 이용 안내 3020
E02 별빛제과 신입사원 온보딩 핸드북 2729


## 청크 수 변경

child_k만 바꾸어 반환되는 부모 ID가 어떻게 달라지는지 확인합니다.


In [4]:
for child_k in (3, 8, 20):
    result = parent_document_search(question["question"], child_k=child_k)
    print(child_k, [item["doc_id"] for item in result])

3 ['E01', 'E02']
8 ['E01', 'E02']
20 ['E01', 'E02', 'M02']
